In [1]:

!pip install scapy


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from scapy.all import sniff, conf
from scapy.layers.inet import IP, TCP, UDP, ICMP
from scapy.layers.l2 import Ether
from datetime import datetime

print('Libraries imported successfully!')
print(f' Default Network Interface: {conf.iface}')

Libraries imported successfully!
 Default Network Interface: \Device\NPF_{54D277B4-AC32-48B9-95DC-0D3E7B9410FF}


In [3]:


PACKET_COUNT = 20       
FILTER       = "tcp"      
LOG_TO_FILE  = True     
LOG_FILE     = "sniffer_log.txt"

# ───────────────────────────────────────────────────────────
print(f' Config set: {PACKET_COUNT} packets | Filter: "{FILTER if FILTER else "all"}"')

 Config set: 20 packets | Filter: "tcp"


In [4]:
PROTOCOL_MAP = {1: "ICMP", 6: "TCP", 17: "UDP"}
packet_number = 0
log_lines = []   


def separator(char="─", length=58):
    return char * length


def interpret_tcp_flags(flags):
    flag_map = {"F": "FIN", "S": "SYN", "R": "RST",
                "P": "PSH", "A": "ACK", "U": "URG"}
    active = [flag_map[f] for f in str(flags) if f in flag_map]
    return " | ".join(active) if active else str(flags)


def interpret_icmp_type(t):
    types = {0: "Echo Reply", 3: "Destination Unreachable",
             5: "Redirect",   8: "Echo Request (Ping)",
             11: "Time Exceeded"}
    return types.get(t, "Unknown")


def extract_payload(packet):
    try:
        raw = bytes(packet.payload.payload.payload)
        if raw:
            return ''.join(chr(b) if 32 <= b < 127 else '.' for b in raw)[:200]
    except Exception:
        pass
    return None


print('Helper functions defined!')

Helper functions defined!


In [5]:
def analyze_packet(packet):
    global packet_number, log_lines
    packet_number += 1
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    output = []

    output.append(separator())
    output.append(f"  Packet #{packet_number}  |  ⏱ {timestamp}")
    output.append(separator())

   
    if packet.haslayer(Ether):
        eth = packet[Ether]
        output.append(f"  [Ethernet]")
        output.append(f"    Src MAC  : {eth.src}")
        output.append(f"    Dst MAC  : {eth.dst}")

  
    if packet.haslayer(IP):
        ip = packet[IP]
        proto = PROTOCOL_MAP.get(ip.proto, f"Unknown ({ip.proto})")
        output.append(f"  [IP]")
        output.append(f"    Src IP   : {ip.src}")
        output.append(f"    Dst IP   : {ip.dst}")
        output.append(f"    Protocol : {proto}")
        output.append(f"    TTL      : {ip.ttl}")
        output.append(f"    Length   : {ip.len} bytes")

       
        if packet.haslayer(TCP):
            tcp = packet[TCP]
            output.append(f"  [TCP]")
            output.append(f"    Src Port : {tcp.sport}")
            output.append(f"    Dst Port : {tcp.dport}")
            output.append(f"    Flags    : {interpret_tcp_flags(tcp.flags)}")

      
        elif packet.haslayer(UDP):
            udp = packet[UDP]
            output.append(f"  [UDP]")
            output.append(f"    Src Port : {udp.sport}")
            output.append(f"    Dst Port : {udp.dport}")
            output.append(f"    Length   : {udp.len} bytes")

      
        elif packet.haslayer(ICMP):
            icmp = packet[ICMP]
            output.append(f"  [ICMP]")
            output.append(f"    Type     : {icmp.type} ({interpret_icmp_type(icmp.type)})")
            output.append(f"    Code     : {icmp.code}")

       
        payload = extract_payload(packet)
        if payload:
            output.append(f"  [Payload]")
            output.append(f"    Data     : {payload}")

    else:
        output.append("  [Non-IP packet]")

    output.append("")

    for line in output:
        print(line)
        log_lines.append(line)


print('Packet analyzer ready!')

Packet analyzer ready!


In [6]:

packet_number = 0
log_lines = []

print('═' * 58)
print('   CodeAlpha — Basic Network Sniffer')
print(f'  Capturing {PACKET_COUNT} packets on: {conf.iface}')
print(f'  Filter: "{FILTER if FILTER else "all traffic"}"')
print('═' * 58)
print()

sniff(
    filter=FILTER,
    prn=analyze_packet,
    count=PACKET_COUNT,
    store=False
)

print('═' * 58)
print(f'   Done! Captured {packet_number} packets.')
print('═' * 58)

══════════════════════════════════════════════════════════
  🔍 CodeAlpha — Basic Network Sniffer
  Capturing 20 packets on: \Device\NPF_{54D277B4-AC32-48B9-95DC-0D3E7B9410FF}
  Filter: "tcp"
══════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────
  📦 Packet #1  |  ⏱ 2026-05-04 20:40:59
──────────────────────────────────────────────────────────
  [Ethernet]
    Src MAC  : fc:01:7c:99:4e:8b
    Dst MAC  : c0:25:e9:53:ff:5a
  [IP]
    Src IP   : 192.168.0.106
    Dst IP   : 4.213.25.241
    Protocol : TCP
    TTL      : 128
    Length   : 83 bytes
  [TCP]
    Src Port : 62289
    Dst Port : 443
    Flags    : PSH | ACK
  [Payload]
    Data     : ....&...........r..M.(...,.....SBJ3.....zL.

──────────────────────────────────────────────────────────
  📦 Packet #2  |  ⏱ 2026-05-04 20:40:59
──────────────────────────────────────────────────────────
  [Ethernet]
    Src MAC  : c0:25:e9:53:ff:5a
    Dst MAC  : fc:01:7c:99:4e:8b
  [I

In [7]:
if LOG_TO_FILE and log_lines:
    with open(LOG_FILE, "w", encoding="utf-8") as f:
        f.write(f"Network Sniffer Log — {datetime.now()}\n")
        f.write(separator() + "\n")
        for line in log_lines:
            f.write(line + "\n")
    print(f'Log saved to "{LOG_FILE}"')
else:
    print('No packets to log or logging is disabled.')

Log saved to "sniffer_log.txt"
